# NLP Assignment — Sentiment Analysis using BERT


In [ ]:
# Cell 1 — Enable GPU
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")


In [ ]:
# Cell 2 — Install Libraries
!pip install -q transformers datasets evaluate accelerate scikit-learn seaborn matplotlib


In [ ]:
# Cell 3 — Import Libraries
import numpy as np
import pandas as pd
import torch
import evaluate
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)


In [ ]:
# Cell 4 — Load Dataset
dataset = load_dataset("stanfordnlp/imdb")

print(dataset)


In [ ]:
# Cell 5 — Display Examples
print("First Training Example:")
print(dataset["train"][0])

print("\nSecond Training Example:")
print(dataset["train"][1])


In [ ]:
# Cell 6 — Load BERT Tokenizer
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully!")
print("Vocabulary size:", tokenizer.vocab_size)


In [ ]:
# Cell 7 — Tokenize Reviews
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

print(tokenized_dataset)


In [ ]:
# Cell 8 — Remove Text Column
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

print(tokenized_dataset)


In [ ]:
# Cell 9 — Create Data Collator
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("Data collator created successfully!")


In [ ]:
# Cell 10 — Load BERT Model
id2label = {
    0: "NEGATIVE",
    1: "POSITIVE"
}

label2id = {
    "NEGATIVE": 0,
    "POSITIVE": 1
}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

print("BERT model loaded successfully!")


In [ ]:
# Cell 11 — Define Metrics
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary"
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


In [ ]:
# Cell 12 — Training Arguments
training_args = TrainingArguments(
    output_dir="./bert-sentiment-model",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_dir="./logs",
    logging_steps=100,
    report_to="none",
    fp16=torch.cuda.is_available()
)


In [ ]:
# Cell 13 — Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer created successfully!")


In [ ]:
# Cell 14 — Train BERT
trainer.train()


In [ ]:
# Cell 15 — Evaluate Model
evaluation_results = trainer.evaluate()

print("\nEvaluation Results:")
for key, value in evaluation_results.items():
    print(f"{key}: {value}")


In [ ]:
# Cell 16 — Generate Predictions
predictions = trainer.predict(tokenized_dataset["test"])

predicted_labels = np.argmax(
    predictions.predictions,
    axis=1
)

actual_labels = predictions.label_ids

print("Predictions generated successfully!")
print("\nFirst 20 predictions:")
print(predicted_labels[:20])
print("\nFirst 20 actual labels:")
print(actual_labels[:20])


In [ ]:
# Cell 17 — Classification Report
print(
    classification_report(
        actual_labels,
        predicted_labels,
        target_names=["NEGATIVE", "POSITIVE"]
    )


In [ ]:
# Cell 18 — Confusion Matrix
cm = confusion_matrix(
    actual_labels,
    predicted_labels
)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"]
)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("BERT Sentiment Analysis - Confusion Matrix")
plt.show()


In [ ]:
# Cell 19 — Prediction Function
def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    device = model.device
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=1
    )

    predicted_class = torch.argmax(
        probabilities,
        dim=1
    ).item()

    confidence = probabilities[
        0, predicted_class
    ].item()

    sentiment = id2label[predicted_class]

    return sentiment, confidence


In [ ]:
# Cell 20 — Test Example Sentences
sentences = [
    "This movie was absolutely amazing. I loved every minute of it.",
    "The movie was terrible and extremely boring.",
    "The acting was excellent and the story was fantastic.",
    "I hated this movie. It was a complete waste of time.",
    "The movie was okay, but nothing special."
]

for sentence in sentences:
    sentiment, confidence = predict_sentiment(sentence)

    print("Sentence:", sentence)
    print("Sentiment:", sentiment)
    print("Confidence:", round(confidence * 100, 2), "%")
    print("-" * 70)


In [ ]:
# Cell 21 — Interactive Prediction
while True:
    text = input("\nEnter a sentence (type 'exit' to stop): ")

    if text.lower() == "exit":
        print("Program terminated.")
        break

    sentiment, confidence = predict_sentiment(text)
    print("\nPredicted Sentiment:", sentiment)
    print("Confidence:", round(confidence * 100, 2), "%")


In [ ]:
# Cell 22 — Save Model
trainer.save_model("./final_bert_sentiment_model")
tokenizer.save_pretrained("./final_bert_sentiment_model")

print("Model saved successfully!")


In [ ]:
# Cell 23 — Load Saved Model
saved_path = "./final_bert_sentiment_model"

loaded_tokenizer = AutoTokenizer.from_pretrained(saved_path)
loaded_model = AutoModelForSequenceClassification.from_pretrained(saved_path)

print("Saved BERT model loaded successfully!")
